**1. Instalação e imports**

In [ ]:
!pip install dbfread unidecode pyarrow

import os
import glob
import gc
from pathlib import Path

import pandas as pd
from dbfread import DBF
from unidecode import unidecode


**2. Paths**


In [ ]:
# Diretório base = pasta onde está o notebook
BASE_DIR = Path(".").resolve()

# Pasta com os DBFs brutos
RAW_DIR = BASE_DIR / "dados_brutos"

# Pasta onde serão salvos os arquivos processados
PROC_DIR = BASE_DIR / "dados_processados"
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Lista de arquivos DBF
dbf_files = [p for p in RAW_DIR.iterdir() if p.suffix.lower() == ".dbf"]
dbf_files

**3. Colunas do SINAN que serão mantidas**

In [ ]:
KEEP_COLS_BASE = [
 "DT_NOTIFIC",
 "NU_ANO",
 "SG_UF_NOT",
 "ID_MUNICIP",
 "ID_UNIDADE",
 "DT_OCOR",
 "NU_IDADE_N",
 "CS_SEXO",
 "CS_GESTANT",
 "CS_RACA",
 "CS_ESCOL_N",
 "SG_UF",
 "ID_MN_RESI",
 "ID_PAIS",
 "DT_INVEST",
 "ID_OCUPA_N",
 "SIT_CONJUG",
 "DEF_TRANS",
 "DEF_FISICA",
 "DEF_MENTAL",
 "DEF_VISUAL",
 "DEF_AUDITI",
 "TRAN_MENT",
 "TRAN_COMP",
 "DEF_OUT",
 "DEF_ESPEC",
 "SG_UF_OCOR",
 "ID_MN_OCOR",
 "HORA_OCOR",
 "LOCAL_OCOR",
 "LOCAL_ESPE",
 "OUT_VEZES",
 "LES_AUTOP",
 "VIOL_FISIC",
 "VIOL_PSICO",
 "VIOL_TORT",
 "VIOL_SEXU",
 "VIOL_TRAF",
 "VIOL_FINAN",
 "VIOL_NEGLI",
 "VIOL_INFAN",
 "VIOL_LEGAL",
 "VIOL_OUTR",
 "VIOL_ESPEC",
 "AG_FORCA",
 "AG_ENFOR",
 "AG_OBJETO",
 "AG_CORTE",
 "AG_QUENTE",
 "AG_ENVEN",
 "AG_FOGO",
 "AG_AMEACA",
 "AG_OUTROS",
 "AG_ESPEC",
 "SEX_ASSEDI",
 "SEX_ESTUPR",
 "CONS_ABORT",
 "CONS_GRAV",
 "CONS_DST",
 "CONS_SUIC",
 "CONS_MENT",
 "CONS_COMP",
 "CONS_ESTRE",
 "CONS_OUTR",
 "CONS_ESPEC",
 "LESAO_NAT",
 "LESAO_ESPE",
 "LESAO_CORP",
 "NUM_ENVOLV",
 "REL_SEXUAL",
 "REL_PAI",
 "REL_MAE",
 "REL_PAD",
 "REL_CONJ",
 "REL_EXCON",
 "REL_NAMO",
 "REL_EXNAM",
 "REL_FILHO",
 "REL_DESCO",
 "REL_IRMAO",
 "REL_CONHEC",
 "REL_CUIDA",
 "REL_PATRAO",
 "REL_INST",
 "REL_POL",
 "REL_PROPRI",
 "REL_OUTROS",
 "REL_ESPEC",
 "AUTOR_SEXO",
 "AUTOR_ALCO",
 "ENC_SAUDE",
 "ENC_TUTELA",
 "ENC_VARA",
 "ENC_ABRIGO",
 "ENC_SENTIN",
 "ENC_DEAM",
 "ENC_DPCA",
 "ENC_DELEG",
 "ENC_MPU",
 "ENC_MULHER",
 "ENC_CREAS",
 "ENC_IML",
 "ENC_OUTR",
 "ENC_ESPEC",
 "REL_TRAB",
 "REL_CAT",
 "CIRC_LESAO",
 "CLASSI_FIN",
 "EVOLUCAO",
 "DT_OBITO",
 "DT_DIGITA",
 "DT_TRANSUS",
 "DT_TRANSDM",
 "DT_TRANSSM",
 "DT_TRANSRM",
 "DT_TRANSRS",
 "DT_TRANSSE",
 "REL_MAD",
 "TPUNINOT",
 "ORIENT_SEX",
 "IDENT_GEN",
 "VIOL_MOTIV",
 "CICL_VID",
 "REDE_SAU",
 "ASSIST_SOC",
 "REDE_EDUCA",
 "ATEND_MULH",
 "CONS_TUTEL",
 "CONS_IDO",
 "DELEG_IDOS",
 "DIR_HUMAN",
 "MPU",
 "DELEG_CRIA",
 "DELEG_MULH",
 "DELEG",
 "INFAN_JUV",
 "DEFEN_PUBL",
 "DT_ENCERRA"
]

**4. Conversão de NU_IDADE_N → idade em anos**

In [ ]:
def idade_em_anos(cod):
    if pd.isna(cod):
        return None
    try:
        cod = int(cod)
    except:
        return None
    
    unidade = cod // 1000
    valor   = cod % 1000
    
    if unidade == 1:   # horas
        return valor / 8760
    if unidade == 2:   # dias
        return valor / 365
    if unidade == 3:   # meses
        return valor / 12
    if unidade == 4:   # anos
        return valor
    return None


**5. Função de processamento**

In [ ]:
def processar_dbf(path_dbf: Path):
    """
    Lê um DBF do SINAN-Violência e filtra:
      • UF = BA
      • município = Salvador (ID_MUNICIP = 292740)
      • sexo = F
      • NÃO violência autoprovocada (LES_AUTOP != '1')
    """
    nome = path_dbf.name

    # extrai ano do nome, ex.: VIOLBR14.dbf -> 2014
    try:
        ano_2d = int(nome[-6:-4])
        ano = 2000 + ano_2d
    except:
        print(f"[AVISO] Não consegui extrair ano de {nome}, pulando.")
        return None

    print(f"\n=== Processando {nome} (ano {ano}) ===")

    tabela = DBF(str(path_dbf), encoding="latin1", ignore_missing_memofile=True)

    registros = []
    total = mantidos = 0

    for rec in tabela:
        total += 1

        sexo    = rec.get("CS_SEXO")
        uf      = rec.get("SG_UF_OCOR")
        mun_raw = rec.get("ID_MN_OCOR")
        les_aut = rec.get("LES_AUTOP")

        # converte município para inteiro, se possível
        mun_ibge = None
        if mun_raw is not None:
            try:
                mun_ibge = int(mun_raw)
            except:
                pass

        # filtros:
        if sexo != "F":          # só mulheres
            continue
        if uf != 29:           # só Bahia
            continue
        if mun_ibge != 292740:   # só Salvador
            continue
        if les_aut == "1":       # exclui violência autoprovocada
            continue

        mantidos += 1
        novo = {}

        for col in KEEP_COLS_BASE:
            if col in tabela.field_names:
                novo[col] = rec.get(col, None)
            else:
                novo[col] = None

        novo["IDADE_ANOS"] = idade_em_anos(rec.get("NU_IDADE_N"))

        registros.append(novo)

    print(f"Total lidos: {total}")
    print(f"Registros mantidos: {mantidos}")

    if not registros:
        return None

    return pd.DataFrame(registros)


**6. Processar todos os anos e salvar em `dados_processados/`**


In [ ]:
for path_dbf in sorted(dbf_files):
    df_ano = processar_dbf(path_dbf)
    if df_ano is None or df_ano.empty:
        continue

    ano = int(df_ano["ano"].iloc[0])
    out_path = PROC_DIR / f"sinan_SSA_mulheres_{ano}.parquet"
    df_ano.to_parquet(out_path, index=False)
    print(f"Salvo: {out_path} -> {df_ano.shape}")

    del df_ano
    gc.collect()

**7. Juntar todos os anos tratados**


In [ ]:
parquet_files = sorted(PROC_DIR.glob("sinan_SSA_mulheres_*.parquet"))
parquet_files

dfs = [pd.read_parquet(f) for f in parquet_files]
sinan = pd.concat(dfs, ignore_index=True)
sinan.shape

sinan.head()

**8. Criar faixa etária**

In [ ]:
bins = [0, 9, 14, 19, 24, 34, 44, 59, 120]
labels = ["0-9", "10-14", "15-19", "20-24", "25-34", "35-44", "45-59", "60+"]

sinan["FAIXA_ETARIA"] = pd.cut(sinan["IDADE_ANOS"], bins=bins, labels=labels)
sinan["FAIXA_ETARIA"].value_counts()

**9. Mapear sexo, raça e tipos de violência**


In [ ]:
map_sexo = {"M": "Masculino", "F": "Feminino", "I": "Ignorado"}
sinan["CS_SEXO_DESC"] = sinan["CS_SEXO"].map(map_sexo)

map_raca = {
    "1": "Branca",
    "2": "Preta",
    "3": "Amarela",
    "4": "Parda",
    "5": "Indígena",
    "9": "Ignorado"
}
sinan["CS_RACA"] = sinan["CS_RACA"].astype(str).str.strip()
sinan["CS_RACA_DESC"] = sinan["CS_RACA"].map(map_raca)

map_viol = {"1": "Sim", "2": "Não", "9": "Ignorado"}

cols_viol = [
    "VIOL_FISIC", "VIOL_PSICO", "VIOL_TORT", "VIOL_SEXU",
    "VIOL_TRAF", "VIOL_FINAN", "VIOL_NEGLI", "VIOL_INFAN",
    "VIOL_LEGAL", "VIOL_OUTR", "VIOL_ESPEC"
]

for col in cols_viol:
    if col in sinan.columns:
        sinan[col] = sinan[col].astype(str).str.strip()
        sinan[col + "_DESC"] = sinan[col].map(map_viol)

def consolidar_tipos(row):
    tipos = []
    if row.get("VIOL_FISIC_DESC") == "Sim": tipos.append("Física")
    if row.get("VIOL_PSICO_DESC") == "Sim": tipos.append("Psicológica")
    if row.get("VIOL_SEXU_DESC") == "Sim": tipos.append("Sexual")
    if row.get("VIOL_TORT_DESC")  == "Sim": tipos.append("Tortura")
    if row.get("VIOL_NEGLI_DESC") == "Sim": tipos.append("Negligência/abandono")
    if row.get("VIOL_FINAN_DESC") == "Sim": tipos.append("Financeira")
    if row.get("VIOL_TRAF_DESC")  == "Sim": tipos.append("Tráfico de pessoas")
    if row.get("VIOL_INFAN_DESC") == "Sim": tipos.append("Contra criança/adolescente")
    if row.get("VIOL_LEGAL_DESC") == "Sim": tipos.append("Violência legal/institucional")
    if row.get("VIOL_OUTR_DESC")  == "Sim": tipos.append("Outras")
    if row.get("VIOL_ESPEC_DESC") == "Sim": tipos.append("Situação específica")
    return ", ".join(tipos)

sinan["TIPO_VIOLENCIA"] = sinan.apply(consolidar_tipos, axis=1)
sinan[["TIPO_VIOLENCIA"]].head()


**10. Exportar dataset final**


In [ ]:
out_parquet = PROC_DIR / "sinan_final_SSA_mulheres.parquet"
out_csv     = PROC_DIR / "sinan_final_SSA_mulheres.csv"

sinan.to_parquet(out_parquet, index=False)
sinan.to_csv(out_csv, sep=";", index=False)

out_parquet, out_csv